# Generate pre-computed image embeddings for faster training/evals

In [1]:
import pandas as pd
import os
import torchvision.transforms as T
import rasterio

import numpy as np
import timm
import torch
import torchgeo.models
from torchgeo.models import ResNet18_Weights, ResNet50_Weights, ViTSmall16_Weights
from tqdm import tqdm

In [2]:
# Configuration Options

root_dir = "/mnt/DATA/leca5365/satclip-s2-1M-2.0/"  # Root directory where the dataset is stored
dataset_csv = "full-index.csv"  # Path to the CSV file containing the dataset

encoder_model_name = "moco_vit16"
crop_size = 224  # Input image size for the encoder model
embed_dim = 256

In [3]:
def load_image(image_path):
    """Load an image from disk and return it as a numpy array."""
    try:
        with rasterio.open(image_path) as f:
            data = f.read().astype(np.float32)
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")
        return None
    return data

def load_encoder(encoder_model_name):
    """Load the specified encoder model."""
    if encoder_model_name == "moco_vit16":
        # Load the MoCo ViT16 model
        print('using pretrained moco vit16')
        weights = ViTSmall16_Weights.SENTINEL2_ALL_MOCO
        in_chans = weights.meta["in_chans"]
        visual = timm.create_model("vit_small_patch16_224", in_chans=in_chans, num_classes=embed_dim)
        visual.load_state_dict(weights.get_state_dict(progress=True), strict=False)
        visual.requires_grad_(False)
        visual.head.requires_grad_(True)
        visual.eval()
        return visual
    else:
        raise ValueError(f"Unsupported encoder model: {encoder_model_name}")

def get_transform(encoder_model_name):
    """Get the appropriate transform for the specified encoder model."""
    if encoder_model_name == "moco_vit16":
        # Define the transform for MoCo ViT16
        def transform(image):
            B10 = np.zeros((1, *image.shape[1:]), dtype=image.dtype)
            image = np.concatenate([image[:10], B10, image[10:]], axis=0)
            image = torch.tensor(image)

            augmentation = T.Compose([
                T.CenterCrop((crop_size, crop_size)),
            ])
            return augmentation(image)
    else:
        raise ValueError(f"Unsupported encoder model: {encoder_model_name}")
    return transform


In [4]:
im = load_image(image_path)

NameError: name 'image_path' is not defined

In [ ]:
type(im)

numpy.ndarray

In [ ]:
TT = get_transform(encoder_model_name)
X = encoder(TT(im).unsqueeze(0).to(device))

In [ ]:
X.cpu()

tensor([[ 0.0364, -0.0458,  0.0018, -0.0346, -0.1306,  0.0263,  0.1181,  0.2469,
          0.3637,  0.0910,  0.0795,  0.1801, -0.0628, -0.1040, -0.2029,  0.2280,
          0.1881, -0.1306, -0.1587, -0.0651, -0.0887,  0.2769,  0.0174, -0.0688,
         -0.2763, -0.1129, -0.0573, -0.3573,  0.0544, -0.0104, -0.0075,  0.3107,
         -0.3400,  0.0536,  0.0348, -0.1326,  0.2434,  0.1682,  0.1834, -0.0583,
          0.1612, -0.1045,  0.0312,  0.0097, -0.2337,  0.1238, -0.0203,  0.1202,
         -0.2189, -0.1333,  0.1644, -0.2115,  0.0647,  0.1117, -0.1098,  0.2096,
          0.0610,  0.2337,  0.2121, -0.1681,  0.0428,  0.1549, -0.1191,  0.1931,
         -0.3978,  0.0496, -0.0573,  0.1539,  0.0946, -0.1063,  0.1044, -0.0774,
         -0.1423, -0.2147,  0.4610,  0.2950,  0.0515, -0.2735,  0.1768, -0.0164,
          0.0521, -0.0229, -0.2703,  0.0080,  0.1210,  0.0677, -0.0634,  0.1188,
         -0.0627, -0.0332,  0.2428, -0.1294, -0.0704, -0.0952,  0.1519,  0.0484,
          0.1566,  0.1403,  

In [5]:
# Load the encoder and dataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = load_encoder(encoder_model_name).to(device)
transform = get_transform(encoder_model_name)

df = pd.read_csv(os.path.join(root_dir, dataset_csv))

image_paths = []
embeddings = []
batch_size = 32

# Iterate through the dataset and compute embeddings for each image
# Modify to use batch processing for efficiency
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Computing embeddings"):
    image_path = os.path.join(root_dir, "images", row['fn'])
    image = transform(load_image(image_path)).unsqueeze(0).to(device)  # Add batch dimension
    embedding = encoder(image)

    embeddings.append(embedding.cpu().detach().numpy())
    image_paths.append(image_path)

# Save embeddings to disk as a parquet file
df_embeddings = pd.DataFrame({'image_path': image_paths, 'embedding': embeddings})
df_embeddings.to_parquet(os.path.join(root_dir, f"{encoder_model_name}_embeddings.parquet"), compression='snappy', index=False)


using pretrained moco vit16


Computing embeddings: 100%|██████████| 1240639/1240639 [2:59:06<00:00, 115.44it/s] 


ArrowInvalid: ('Can only convert 1-dimensional array values', 'Conversion failed for column embedding with type object')

In [10]:
embeddings_array = np.vstack(embeddings)

In [13]:
df_embeddings = pd.DataFrame({'image_path': image_paths, 'embedding': embeddings_array.tolist()})
df_embeddings.to_parquet(os.path.join(root_dir, f"{encoder_model_name}_embeddings.parquet"), compression='snappy', index=False)


In [12]:
embeddings_array.tolist()[0]

[[0.17337334156036377,
  0.02107492834329605,
  -0.047085512429475784,
  0.3564263582229614,
  0.0395788848400116,
  0.1651160567998886,
  -0.08178205788135529,
  0.0691879540681839,
  -0.17494520545005798,
  0.14239387214183807,
  0.13826505839824677,
  0.2634410858154297,
  0.03873710334300995,
  0.16981898248195648,
  -0.08638308942317963,
  0.23832069337368011,
  0.10306323319673538,
  -0.14971260726451874,
  0.07315842807292938,
  -0.37219393253326416,
  -0.05702657997608185,
  0.18619780242443085,
  -0.012152016162872314,
  -0.0852709636092186,
  -0.04888123273849487,
  0.12427981197834015,
  0.2584210932254791,
  0.4148259162902832,
  -0.3242889642715454,
  -0.3437372148036957,
  -0.07738769799470901,
  -0.027946054935455322,
  -0.07176893949508667,
  -0.027241013944149017,
  -0.03834620118141174,
  -0.0344868004322052,
  0.2244034856557846,
  -0.10009030997753143,
  0.11607015132904053,
  -0.21186646819114685,
  0.36570003628730774,
  -0.2599866986274719,
  0.1452341079711914,


In [ ]:
dft = pd.read_parquet(os.path.join(root_dir, f"{encoder_model_name}_embeddings.parquet"))